In [1]:
import numpy as np
import pandas as pd

speech_df = pd.read_csv("speech.csv")

In [2]:
# Print the first 5 rows of the text col

speech_df['text'].head()

0    Fellow-Citizens of the Senate and of the House...
1    Fellow Citizens:  I AM again called upon by th...
2    WHEN it was first perceived, in early times, t...
3    Friends and Fellow-Citizens:  CALLED upon to u...
4    PROCEEDING, fellow-citizens, to that qualifica...
Name: text, dtype: object

In [3]:
# Replace all non letter characters with a whitespace
speech_df['text_clean'] = speech_df['text'].str.replace('[^a-zA-Z]', ' ', regex=True)

# Change to lower case
speech_df['text_clean'] = speech_df['text_clean'].str.lower()

# Print the first 5 rows of the text_clean column
print(speech_df['text_clean'].head())

0    fellow citizens of the senate and of the house...
1    fellow citizens   i am again called upon by th...
2    when it was first perceived  in early times  t...
3    friends and fellow citizens   called upon to u...
4    proceeding  fellow citizens  to that qualifica...
Name: text_clean, dtype: object


In [4]:
# Find the length of each text
speech_df['char_cnt'] = speech_df['text_clean'].str.len()

# Count the number of words in each text
speech_df['word_cnt'] = speech_df['text_clean'].str.split().str.len()

# Find the average length of word
speech_df['avg_word_length'] = speech_df['char_cnt'] / speech_df['word_cnt']

# Print the first 5 rows of these columns
print(speech_df[['text_clean', 'char_cnt', 'word_cnt', 'avg_word_length']])

                                           text_clean  char_cnt  word_cnt  \
0   fellow citizens of the senate and of the house...      8616      1432   
1   fellow citizens   i am again called upon by th...       787       135   
2   when it was first perceived  in early times  t...     13871      2323   
3   friends and fellow citizens   called upon to u...     10144      1736   
4   proceeding  fellow citizens  to that qualifica...     12902      2169   
5   unwilling to depart from examples of the most ...      7003      1179   
6   about to add the solemnity of an oath to the o...      7148      1211   
7   i should be destitute of feeling if i was not ...     19894      3382   
8   fellow citizens   i shall not attempt to descr...     26322      4466   
9   in compliance with an usage coeval with the ex...     17753      2922   
10  fellow citizens   about to undertake the arduo...      6818      1130   
11  fellow citizens   the will of the american peo...      7061      1179   

- Transformation text


| Feature      | CountVectorizer                         | TfidfVectorizer                                      |
| ------------ | --------------------------------------- | ---------------------------------------------------- |
| What it does | Counts how many times each word appears | Counts words and weights them by importance (TF-IDF) |
| Output       | Word counts (integers)                  | TF-IDF scores (floats, usually 0–1)                  |
| Common words | Get high counts                         | Get low scores (penalized)                           |
| Rare words   | Get low counts                          | Get high scores (emphasized)                         |
| Best for     | When all words are equally important    | When rare words are more important                   |


##### CountVectorizer

In [5]:
# Import CountVectorizer
from sklearn.feature_extraction.text import CountVectorizer

# Instantiate CountVectorizer
cv = CountVectorizer()

# Fit the vectorizer
cv.fit(speech_df['text_clean'])

# Print feature names
print(cv.get_feature_names_out())

['abandon' 'abandoned' 'abandonment' ... 'zealous' 'zealously' 'zone']


In [6]:
# Apply the vectorizer
cv_transformed = cv.transform(speech_df['text_clean'])

# Print the full array
cv_array = cv_transformed.toarray()

# Print the shape of cv_array
print(cv_array.shape)

(58, 9043)


In [7]:
# Import CountVectorizer
from sklearn.feature_extraction.text import CountVectorizer

# Specify arguments to limit the number of features generated
cv = CountVectorizer(min_df = 0.2, max_df = 0.8)

# Fit, transform, and convert into array
cv_transformed = cv.fit_transform(speech_df['text_clean'])
cv_array = cv_transformed.toarray()

# Print the array shape
print(cv_array.shape)

(58, 818)


In [8]:
# Create a DataFrame with these features
cv_df = pd.DataFrame(cv_array, 
                     columns=cv.get_feature_names_out()).add_prefix('Counts_')

# Add the new columns to the original DataFrame
speech_df_new = pd.concat([speech_df, cv_df], axis=1, sort=False)
print(speech_df_new.head())

                Name         Inaugural Address                      Date  \
0  George Washington   First Inaugural Address  Thursday, April 30, 1789   
1  George Washington  Second Inaugural Address     Monday, March 4, 1793   
2         John Adams         Inaugural Address   Saturday, March 4, 1797   
3   Thomas Jefferson   First Inaugural Address  Wednesday, March 4, 1801   
4   Thomas Jefferson  Second Inaugural Address     Monday, March 4, 1805   

                                                text  \
0  Fellow-Citizens of the Senate and of the House...   
1  Fellow Citizens:  I AM again called upon by th...   
2  WHEN it was first perceived, in early times, t...   
3  Friends and Fellow-Citizens:  CALLED upon to u...   
4  PROCEEDING, fellow-citizens, to that qualifica...   

                                          text_clean  char_cnt  word_cnt  \
0  fellow citizens of the senate and of the house...      8616      1432   
1  fellow citizens   i am again called upon by th...  

##### TfidfVectorizer

In [9]:
# Import TfidfVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

# Instantiate TfidfVectorizer (limit feature to 100, and remove engilsh stop wrod)
tv = TfidfVectorizer(max_features = 100,  stop_words='english')

# Fit the vectorizer and transform the data
tv_transformed = tv.fit_transform(speech_df['text_clean'])

# Create a DataFrame with these features
tv_df = pd.DataFrame(tv_transformed.toarray(), 
                     columns=tv.get_feature_names_out()).add_prefix('TFIDF_')
print(tv_df.head())

   TFIDF_action  TFIDF_administration  TFIDF_america  TFIDF_american  \
0      0.000000              0.133415       0.000000        0.105388   
1      0.000000              0.261016       0.266097        0.000000   
2      0.000000              0.092436       0.157058        0.073018   
3      0.000000              0.092693       0.000000        0.000000   
4      0.041334              0.039761       0.000000        0.031408   

   TFIDF_americans  TFIDF_believe  TFIDF_best  TFIDF_better  TFIDF_change  \
0              0.0       0.000000    0.000000      0.000000      0.000000   
1              0.0       0.000000    0.000000      0.000000      0.000000   
2              0.0       0.000000    0.026112      0.060460      0.000000   
3              0.0       0.090942    0.117831      0.045471      0.053335   
4              0.0       0.000000    0.067393      0.039011      0.091514   

   TFIDF_citizens  ...  TFIDF_things  TFIDF_time  TFIDF_today  TFIDF_union  \
0        0.229644  ...    

In [10]:
# Isolate the row to be examined
sample_row = tv_df.iloc[0]

# Print the top 5 words of the sorted output
print(sample_row.sort_values(ascending=False).head())

TFIDF_government    0.367430
TFIDF_public        0.333237
TFIDF_present       0.315182
TFIDF_duty          0.238637
TFIDF_citizens      0.229644
Name: 0, dtype: float64


In [11]:
train_speech_df = speech_df.iloc[:45]
test_speech_df = speech_df.iloc[45:]

In [12]:
# Instantiate TfidfVectorizer
tv = TfidfVectorizer(max_features=100, stop_words='english')

# Fit the vectorizer and transform the data
tv_transformed = tv.fit_transform(speech_df['text_clean'])

# Transform test data
test_tv_transformed = tv.transform(test_speech_df['text_clean'])

# Create new features for the test set
test_tv_df = pd.DataFrame(test_tv_transformed.toarray(), 
                          columns=tv.get_feature_names_out()).add_prefix('TFIDF_')
print(test_tv_df.head())

   TFIDF_action  TFIDF_administration  TFIDF_america  TFIDF_american  \
0      0.000000              0.031440       0.192310        0.074505   
1      0.000000              0.000000       0.472459        0.034865   
2      0.000000              0.000000       0.102291        0.118890   
3      0.032737              0.062983       0.192627        0.024876   
4      0.000000              0.000000       0.182080        0.141084   

   TFIDF_americans  TFIDF_believe  TFIDF_best  TFIDF_better  TFIDF_change  \
0         0.039503       0.061692    0.000000      0.092538      0.036181   
1         0.000000       0.000000    0.037405      0.151563      0.000000   
2         0.000000       0.049222    0.127550      0.098444      0.000000   
3         0.356111       0.308970    0.053376      0.000000      0.000000   
4         0.224408       0.050066    0.086492      0.100132      0.029362   

   TFIDF_citizens  ...  TFIDF_things  TFIDF_time  TFIDF_today  TFIDF_union  \
0        0.021647  ...    

In [13]:
# Instantiate TfidfVectorizer
tv = TfidfVectorizer(max_features=100, stop_words='english')

# Fit the vectroizer and transform the data
tv_transformed = tv.fit_transform(train_speech_df['text_clean'])

# Transform test data
test_tv_transformed = tv.transform(test_speech_df['text_clean'])

# Create new features for the test set
test_tv_df = pd.DataFrame(test_tv_transformed.toarray(), 
                          columns=tv.get_feature_names_out()).add_prefix('TFIDF_')
print(test_tv_df.head())

   TFIDF_action  TFIDF_administration  TFIDF_america  TFIDF_american  \
0      0.000000              0.029540       0.233954        0.082703   
1      0.000000              0.000000       0.547457        0.036862   
2      0.000000              0.000000       0.126987        0.134669   
3      0.037094              0.067428       0.267012        0.031463   
4      0.000000              0.000000       0.221561        0.156644   

   TFIDF_authority  TFIDF_best  TFIDF_business  TFIDF_citizens  \
0         0.000000    0.000000        0.000000        0.022577   
1         0.000000    0.036036        0.000000        0.015094   
2         0.000000    0.131652        0.000000        0.000000   
3         0.039990    0.061516        0.050085        0.077301   
4         0.028442    0.087505        0.000000        0.109959   

   TFIDF_commerce  TFIDF_common  ...  TFIDF_subject  TFIDF_support  \
0             0.0      0.000000  ...            0.0       0.000000   
1             0.0      0.00000

In [14]:
# --- N-GRAM ANALYSIS ---
# Bigrams (2-word combinations)
cv_bigram = CountVectorizer(max_features=100, stop_words='english', ngram_range=(2, 2))
cv_bigram_transformed = cv_bigram.fit_transform(speech_df['text_clean'])
cv_bigram_df = pd.DataFrame(cv_bigram_transformed.toarray(), 
                            columns=cv_bigram.get_feature_names_out()).add_prefix('Bigram_')
print("Bigram features:")
print(cv_bigram_df.head())

# Trigrams (3-word combinations)
cv_trigram = CountVectorizer(max_features=50, stop_words='english', ngram_range=(3, 3))
cv_trigram_transformed = cv_trigram.fit_transform(speech_df['text_clean'])
cv_trigram_df = pd.DataFrame(cv_trigram_transformed.toarray(), 
                             columns=cv_trigram.get_feature_names_out()).add_prefix('Trigram_')
print("\nTrigram features:")
print(cv_trigram_df.head())

Bigram features:
   Bigram_administration government  Bigram_almighty god  \
0                                 0                    0   
1                                 1                    0   
2                                 0                    0   
3                                 0                    0   
4                                 0                    0   

   Bigram_american people  Bigram_beloved country  Bigram_best ability  \
0                       2                       0                    0   
1                       0                       0                    0   
2                       3                       0                    0   
3                       0                       1                    0   
4                       0                       0                    0   

   Bigram_best interests  Bigram_branch government  Bigram_carry effect  \
0                      0                         0                    0   
1                      0   

In [15]:
# --- COMBINE BOTH VECTORIZERS ---
# Combine CountVectorizer and TfidfVectorizer features
combined_df = pd.concat([cv_df, tv_df], axis=1)
print(f"Combined shape: {combined_df.shape}")
print(combined_df.head())

Combined shape: (58, 918)
   Counts_abiding  Counts_ability  Counts_able  Counts_about  Counts_above  \
0               0               0            0             0             0   
1               0               0            0             1             0   
2               0               0            0             0             0   
3               0               0            0             1             1   
4               0               0            1             0             0   

   Counts_abroad  Counts_accept  Counts_accomplished  Counts_achieve  \
0              0              0                    1               0   
1              0              0                    0               0   
2              1              0                    0               0   
3              1              0                    0               0   
4              0              0                    0               0   

   Counts_across  ...  TFIDF_things  TFIDF_time  TFIDF_today  TFIDF_unio

In [16]:
# --- COMBINED FEATURE ANALYSIS ---
# After combining CountVectorizer and TfidfVectorizer features
combined_df = pd.concat([cv_df, tv_df], axis=1)

# Analyze both types of features
print("\n=== Combined Feature Analysis ===\n")

# 1. TF-IDF top words
tfidf_cols = [col for col in combined_df.columns if col.startswith('TFIDF_')]
tfidf_sums = combined_df[tfidf_cols].sum().sort_values(ascending=False)
print("Top 10 TF-IDF words:")
for word, score in tfidf_sums.head(10).items():
    print(f"  {word.replace('TFIDF_', '')}: {score:.4f}")

# 2. Count top words
count_cols = [col for col in combined_df.columns if col.startswith('Counts_')]
count_sums = combined_df[count_cols].sum().sort_values(ascending=False)
print("\nTop 10 Count words:")
for word, score in count_sums.head(10).items():
    print(f"  {word.replace('Counts_', '')}: {score:.0f}")

# 3. Comparison of rankings
print("\n=== Word Ranking Comparison ===")
tfidf_top5 = set(tfidf_sums.head(5).index.str.replace('TFIDF_', ''))
count_top5 = set(count_sums.head(5).index.str.replace('Counts_', ''))
common = tfidf_top5 & count_top5
print(f"Words in both top 5: {common}")


=== Combined Feature Analysis ===

Top 10 TF-IDF words:
  government: 11.7505
  people: 11.2659
  shall: 7.9591
  nation: 7.2835
  world: 7.1751
  america: 6.7919
  country: 6.7818
  states: 6.7775
  great: 6.5397
  public: 6.2456

Top 10 Count words:
  states: 326
  should: 324
  peace: 254
  public: 226
  america: 219
  such: 215
  constitution: 205
  under: 192
  you: 192
  freedom: 188

=== Word Ranking Comparison ===
Words in both top 5: set()


In [17]:
# --- TOP WORDS SUMMARY TABLE ---
# Create a summary of top words across all documents
word_counts = tv_df.sum().sort_values(ascending=False)
top_20_words = word_counts.head(20)
print("\nTop 20 words across all speeches:")
print(top_20_words)


Top 20 words across all speeches:
TFIDF_government      11.750535
TFIDF_people          11.265916
TFIDF_shall            7.959075
TFIDF_nation           7.283539
TFIDF_world            7.175133
TFIDF_america          6.791895
TFIDF_country          6.781753
TFIDF_states           6.777489
TFIDF_great            6.539715
TFIDF_public           6.245572
TFIDF_peace            6.225727
TFIDF_new              5.639707
TFIDF_freedom          5.193071
TFIDF_union            5.139781
TFIDF_power            5.121458
TFIDF_citizens         5.047634
TFIDF_constitution     5.027740
TFIDF_war              4.782031
TFIDF_united           4.754359
TFIDF_let              4.695957
dtype: float64
